# CliniBridge: Multi-Agent Clinical Context Ingestion & Demonstration

This notebook allows you to interactively run the **ClinicalBridge** (CliniBridge) multi-agent pipeline on the 5 clinical scenarios. It loads the configuration, initializes the LangGraph orchestrator, and executes the runs.

### 0. Optional: Install Required Libraries
If you haven't installed the required packages in your active environment yet, run the cell below to install them directly from this notebook:

In [ ]:
# Run this cell to install all dependencies directly in your current Jupyter kernel
# !pip install -r requirements.txt ipykernel

### 1. Setup and Environmental Checks
First, we make sure that the environment is loaded, paths are configured, and the OpenRouter API key is present.

In [ ]:
import os
import sys
import json
from dotenv import load_dotenv

# Ensure project root is in the path
cwd = os.getcwd()
BASE_DIR = cwd if os.path.exists(os.path.join(cwd, "Orchestrator")) else os.path.join(cwd, "CliniBridge")
if not os.path.exists(BASE_DIR):
    parent = os.path.dirname(cwd)
    BASE_DIR = parent if os.path.exists(os.path.join(parent, "Orchestrator")) else os.path.join(parent, "CliniBridge")

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

# Load .env file from multiple potential locations
env_paths = [os.path.join(BASE_DIR, ".env"), os.path.join(os.path.dirname(BASE_DIR), ".env")]
for path in env_paths:
    if os.path.exists(path):
        load_dotenv(path, override=True)
        print(f"✓ Environment loaded from {path}")
        break

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key or "your_openrouter" in api_key:
    print("WARNING: OPENROUTER_API_KEY is missing or invalid in .env. Running the orchestrator will fail.")
else:
    print("✓ API key is configured.")

### 2. Initialize the Orchestrator
Now we import the `Orchestrator` class and create an instance. This compiles the LangGraph state machine and saves the workflow diagram to `graph.png`.

In [ ]:
from Orchestrator.orchestrator import Orchestrator

print("Initializing Orchestrator and compiling LangGraph...")
orchestrator = Orchestrator()
print("✓ Orchestrator compiled successfully. Graph saved to graph.png.")

### 3. Scenario Execution Helper
This helper loads a scenario JSON alert, runs it through the orchestrator, and pretty-prints the resulting **Clinical Context Brief**.

In [ ]:
def run_scenario(scenario_number):
    scenario_file = f"scenario_{scenario_number}.json"
    scenario_path = os.path.join(BASE_DIR, "data", "alerts", scenario_file)
    
    if not os.path.exists(scenario_path):
        print(f"Error: Scenario file {scenario_file} not found at {scenario_path}")
        return
        
    with open(scenario_path, "r", encoding="utf-8") as f:
        alert = json.load(f)
        
    print(f"==================================================")
    print(f" RUNNING: Scenario {scenario_number} - {alert.get('description')}")
    print(f"==================================================")
    print(f"Patient: {alert.get('patient_id')} | Reading: {alert.get('reading')} = {alert.get('value')} {alert.get('unit')}")
    print("Running orchestrator... (this may take 10-30 seconds depending on LLM response times)")
    
    state = orchestrator.run(alert)
    brief = state.get("clinical_brief")
    
    print(f"\n--------------------------------------------------")
    print(f" CLINICAL BRIEF OUTPUT (PDF saved in patient logs)")
    print(f"--------------------------------------------------")
    print(json.dumps(brief, indent=2, ensure_ascii=False))
    print(f"==================================================\n")

### 4. Interactive Testing
You can run the scenarios one by one below. Change the argument to `run_scenario()` to test different cases:
*   **Scenario 1**: Missed Medication (systolic BP spike to 178/108 due to cough side-effect discontinuation).
*   **Scenario 2**: False Alarm (glucose spike to 210 due to high carb meal test).
*   **Scenario 3**: Silent Deterioration (gradual weight gain in heart failure indicating fluid retention).
*   **Scenario 4**: Incomplete Record (BP spike in a transfer patient with sparse history).
*   **Scenario 5**: Conflicting Data (compliance claim vs sub-therapeutic labs).

In [ ]:
# Run Scenario 1 (Missed Medication - Expects High Severity/Emergency bypass)
run_scenario(1)

In [ ]:
# Run Scenario 2 (False Alarm - Expects Moderate Severity/Full synthesis)
run_scenario(2)

In [ ]:
# Run Scenario 3 (Silent Deterioration - Expects Elevated Severity/Full synthesis)
run_scenario(3)

In [ ]:
# Run Scenario 4 (Incomplete Record - Expects High Triage due to safety/sparse checks)
run_scenario(4)

In [ ]:
# Run Scenario 5 (Conflicting Data - Expects Elevated Triage/Conflicting check synthesis)
run_scenario(5)